In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh
from scipy.ndimage import median_filter

from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
from skimage import measure

from scipy.ndimage import uniform_filter1d

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline

base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/"

out_folder = "Meshes/extracted_lines/"


folder_g = "B_FFE i0.9 XD Fast Gch/"
folder_l = "B_FFE i0.9 XD Fast Lateral/"
folder_s = "cropped T1/"

segment_folder_g = "segment/"
segment_folder_l = "segment/"
segment_folder_s = "segment_w_lens/"

pixel_spacing_s = "pixel_spacing.txt"
pixel_spacing_g = "1601_pixel_spacing.txt"
pixel_spacing_l = "1401_pixel_spacing.txt"

file_s = "segmentations_pdt1_v160725.seg.nrrd"
file_l = "Segmentation.seg.nrrd"
file_g = "Segmentation_4.seg.nrrd"

lens_id_s = 2
lens_id_l = 2
lens_id_g = 6



In [3]:
folders = [base + folder_s , base + folder_l, base + folder_g]
files = [base + folder_s + segment_folder_s + file_s, base + folder_l + segment_folder_l + file_l, base + folder_g + segment_folder_g + file_g ]
lens_ids = [lens_id_s, lens_id_l, lens_id_g]
pixel_spacing_files = [base + folder_s + pixel_spacing_s, base + folder_l + pixel_spacing_l, base + folder_g + pixel_spacing_g]
names = ["centre", "lat", "med"]


In [5]:
lens_masks = []
pixel_spacings = []
offsets = []

for i in range(3):
    data, header = nrrd.read(files[i])
    pixel_spacing = numpy.loadtxt(pixel_spacing_files[i])
    pixel_spacings.append(pixel_spacing[0])

    print(pixel_spacing)

    offsets.append( numpy.loadtxt(folders[i] + "Meshes/offset.txt" ))


    if pixel_spacing[2]!= pixel_spacing[1]:
        factor = pixel_spacing[2]/pixel_spacing[1]

    data = ndimage.zoom(data, [1,1,factor], order = 0)

    mask = numpy.zeros(data.shape, dtype=numpy.uint8)
    mask[(data>lens_ids[i]-0.5) & (data<lens_ids[i]+0.5 )]=1

    lens_masks.append(mask)




[0.3125 0.3125 0.5   ]
[0.75       0.75       0.75000042]
[0.75       0.75       0.75000042]


## lens axis

In [6]:
axis_directions = []


for i in range(3):

    mask_points = numpy.argwhere(lens_masks[i] == 1)

    print("mask_points = ", mask_points.shape)

    # 2. Calculate the center of mass (mean of the coordinates)
    lens_center = mask_points.mean(axis=0)

    # 3. Center the coordinates around the mean
    X = mask_points - lens_center

    # 4. Run SVD to get the principal axes
    _, _, V_T = numpy.linalg.svd(X)

    # The first row of V^T is the primary axis direction
    axis_direction = V_T[-1, :]  # Shape is now (3,) or (2,)


    axis_directions.append(axis_direction)

    def axis(t):
        return lens_center + t * axis_direction
    axis_points = [axis(-20.0), axis(20.0)]

    export_centerline(axis_points, pixel_spacings[i] , folders[i] + out_folder + "lens_axis_"+names[i]+".vtp", offsets[i])


mask_points =  (2778, 3)
mask_points =  (534, 3)
mask_points =  (378, 3)


## Angles of gaze

In [7]:
def angle_between_vectors(v1, v2):
    dot_product = numpy.dot(v1, v2)
    
    norm_v1 = numpy.linalg.norm(v1)
    norm_v2 = numpy.linalg.norm(v2)
    
    cos_theta = dot_product / (norm_v1 * norm_v2)
    

    # Get the angle in radians
    angle_radians = numpy.arccos(cos_theta)
    
    # Convert to degrees
    angle_degrees = numpy.degrees(angle_radians)
    
    if angle_degrees>90:
        angle_degrees = 180-angle_degrees
        angle_radians = numpy.pi - angle_radians
    return angle_degrees, angle_radians

angle_l, angle_l_rad = angle_between_vectors(axis_directions[0], axis_directions[1])
print(f"Angle centre vs l: {angle_l}")

angle_g, angle_g_rad = angle_between_vectors(axis_directions[0], axis_directions[2])
print(f"Angle centre vs g: {angle_g}")

Angle centre vs l: 31.991662112463953
Angle centre vs g: 41.11785932148172


In [ ]:

import trimesh

# 1. Load the STL mesh
mesh_obj = trimesh.load_mesh("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/re-meshed_h_0-9/id_3_shell__h_0-9.stl")



# 3. Create the 4x4 transformation matrix for rotation around an arbitrary point
transform_matrix = trimesh.transformations.rotation_matrix(
    angle=angle_g_rad,
    direction=axis_directions[0],
    point=[0,0,0]
)

# 4. Apply the matrix transformation directly to the mesh
mesh_obj.apply_transform(transform_matrix)

# 5. Export back to STL
mesh_obj.export(folders[0]+ "Meshes/rotated_gaze.stl")


b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00n*\x00\x00?FB>\x1c\x9bm\xbf+\xf5\xa3\xbeP?\xa1?\xbfi=\xc1\xcfMA\xc0\x16\xfb\x08@\xdf\x14<\xc17Z/\xc0\x1c\xd9\xa9?FOB\xc1\xcd\xfc\x05\xc0\x00\x00\xad\xc2a?q\\\xe8>\x9e\xdc\x02>"\r3A(\xe5\x88@h\x01;@\xdd\xd5:A5 `@\x13\x83\x14@\x8b\xc74As\xc5\x89@\x81\x14\x05@\x00\x00\x96v\xac>\x8b@\x1e\xbfk\xd05\xbf\xf51\x91@-O\xb8\xc0\xbb\x0f&\xc1,\x8f\xaa@hn\xae\xc0qX$\xc1kS\xa1@\xcb[\xc8\xc0`@\x1b\xc1\x00\x00\x1a\xb6\xd3>b\xadg\xbf\x1d\xef\xcc\xbd\xbe\xf8c@\xf0\x169\xc1\x8d\x93\x80\xbf=\x85|@\x91|4\xc1\']\x01\xc0\x0e\xd8\x8b@\xe8\xab2\xc1\t\x86\xa5\xbf\x00\x00\x06w\xbf>\xcd\xb9f\xbfH\x08`\xbe\xa4L\x92@\xb40-\xc1\xc4\x165\xc0=\x85|@\x91|4\xc1\']\x01\xc0.Al@U\xe41\xc1\xd7\xebG\xc0\x00\x0